# EX_12 — Contexto largo y multimodal (ejercicios)

**Notebook de referencia:** `notebook/12_Modelos_Contexto_Multimodales.ipynb`

**Tiempo orientativo:** ~30 minutos.


## Actividad 1 — Presupuesto de tokens

Estima (orden de magnitud) tokens para: 10 páginas de texto, 1 imagen 1024×1024 en un modelo que la trata como patches, y 2 minutos de audio crudo 16 kHz 16-bit. Usa markdown con supuestos explícitos.


### Actividad 1 — Presupuesto de tokens

Para estimar el presupuesto de tokens, se hacen los siguientes supuestos simplificados:

* En texto, se asume aproximadamente **1 token por cada 3–4 caracteres** o unas **700–800 tokens por página** de texto estándar.
* En imagen, se asume que el modelo divide la imagen en **patches** de tamaño aproximado `16×16 píxeles`.
* En audio, se considera audio crudo PCM a **16 kHz**, **16-bit**, mono.

#### 1. Diez páginas de texto

Si una página contiene unas 500 palabras, 10 páginas tendrían aproximadamente:

```text
10 páginas × 500 palabras = 5.000 palabras
```

Como una palabra suele ocupar algo más de un token en muchos tokenizadores, el orden de magnitud sería:

```text
≈ 6.000 – 8.000 tokens
```

Por tanto, 10 páginas de texto estarían en el orden de:

```text
10^4 tokens
```

#### 2. Una imagen de 1024×1024 tratada como patches

Si el modelo divide la imagen en patches de `16×16 píxeles`:

```text
1024 / 16 = 64 patches por lado
64 × 64 = 4.096 patches
```

Si cada patch se representa aproximadamente como un token visual o unidad de entrada, la imagen tendría:

```text
≈ 4.000 tokens visuales
```

Por tanto, una imagen 1024×1024 estaría en el orden de:

```text
10^3 – 10^4 tokens visuales
```

El número exacto dependería del modelo, del tamaño de patch y de si aplica compresión o reducción previa.

#### 3. Dos minutos de audio crudo 16 kHz 16-bit

Dos minutos equivalen a:

```text
2 minutos = 120 segundos
```

Con una frecuencia de muestreo de 16.000 muestras por segundo:

```text
120 × 16.000 = 1.920.000 muestras
```

Como cada muestra ocupa 16 bits, es decir, 2 bytes:

```text
1.920.000 × 2 = 3.840.000 bytes
```

Si el audio crudo se tratara de forma ingenua como bytes o texto codificado, el coste sería muy alto, del orden de:

```text
≈ 10^6 tokens o unidades de entrada
```

En la práctica, los modelos de audio no suelen introducir audio crudo directamente como texto, sino que usan codificadores, espectrogramas o frames de audio, por lo que el número real de tokens dependería mucho de la arquitectura del modelo.

### Resumen aproximado

| Entrada                            |                                                  Estimación | Orden de magnitud |
| ---------------------------------- | ----------------------------------------------------------: | ----------------: |
| 10 páginas de texto                |                                          6.000–8.000 tokens |            `10^4` |
| Imagen 1024×1024 con patches 16×16 |                                     ≈ 4.096 tokens visuales |       `10^3–10^4` |
| Audio crudo 2 min, 16 kHz, 16-bit  | ≈ 3,84 MB / hasta millones de unidades si se tokeniza crudo |            `10^6` |

La conclusión principal es que el coste de entrada depende mucho de la modalidad. El texto suele ser relativamente manejable, la imagen depende del número de patches y el audio crudo puede ser muy costoso si no se transforma previamente mediante un encoder especializado.


## Actividad 2 — Estrategia de ventana

Describe cómo partirías un documento de 200k tokens para un modelo de 128k de ventana (resumen jerárquico, índice, etc.). Respuesta en español.


### Actividad 2 — Estrategia de ventana para un documento de 200k tokens

Si tengo un documento de aproximadamente **200.000 tokens** y el modelo solo admite una ventana de contexto de **128.000 tokens**, no intentaría introducir el documento completo de una sola vez. Aunque 128k es una ventana grande, sigue siendo insuficiente para procesar todo el documento junto con instrucciones, pregunta del usuario y espacio para la respuesta.

Una estrategia adecuada sería dividir el documento en partes y aplicar un enfoque de **resumen jerárquico e indexación**.

#### 1. División inicial del documento

Primero dividiría el documento en bloques manejables, por ejemplo de entre **10.000 y 20.000 tokens** cada uno. Cada bloque debería respetar la estructura natural del documento: capítulos, secciones, apartados o encabezados. Es preferible no cortar el texto en medio de una idea importante.

Por ejemplo:

```text
Documento completo: 200.000 tokens

Bloque 1: 0–20.000 tokens
Bloque 2: 20.000–40.000 tokens
Bloque 3: 40.000–60.000 tokens
...
Bloque 10: 180.000–200.000 tokens
```

#### 2. Resumen local de cada bloque

Después generaría un resumen de cada bloque. Cada resumen debería conservar:

* ideas principales,
* conceptos técnicos importantes,
* entidades relevantes,
* decisiones o conclusiones,
* citas o referencias internas,
* posibles preguntas que ese bloque podría responder.

El objetivo no sería comprimir sin criterio, sino crear una representación útil para recuperación posterior.

#### 3. Creación de un índice del documento

Además de los resúmenes, construiría un índice con metadatos de cada bloque:

```text
Bloque 1: tema principal, secciones incluidas, palabras clave
Bloque 2: tema principal, secciones incluidas, palabras clave
Bloque 3: tema principal, secciones incluidas, palabras clave
...
```

Este índice permitiría seleccionar solo las partes relevantes para una consulta concreta, en lugar de cargar todo el documento.

#### 4. Resumen jerárquico

A continuación, combinaría los resúmenes locales en resúmenes de nivel superior. Por ejemplo:

```text
Nivel 1: resumen de cada bloque
Nivel 2: resumen de grupos de bloques
Nivel 3: resumen global del documento
```

Así se obtiene una visión general del documento sin necesidad de introducir los 200k tokens completos en la ventana del modelo.

#### 5. Recuperación selectiva según la pregunta

Cuando el usuario haga una pregunta, usaría el índice o embeddings para identificar los bloques más relevantes. Solo esos fragmentos, junto con sus resúmenes y metadatos, se introducirían en la ventana de contexto.

Por ejemplo, ante una pregunta específica, el sistema podría cargar:

```text
- Resumen global del documento
- Índice de secciones
- 3 o 4 bloques relevantes
- Fragmentos exactos necesarios para citar o justificar la respuesta
```

De esta forma, la entrada total se mantiene por debajo del límite de 128k tokens.

#### 6. Uso de ventana larga con margen de seguridad

Aunque el modelo permita 128k tokens, no conviene ocupar toda la ventana. Dejaría margen para:

* instrucciones del sistema,
* pregunta del usuario,
* documentos recuperados,
* razonamiento interno del flujo,
* respuesta final.

Por ejemplo, intentaría no superar unos **90k–100k tokens de contexto efectivo**, dejando el resto para la generación.

#### Conclusión

Para un documento de 200k tokens usaría una estrategia combinada de **chunking, resumen jerárquico, índice semántico y recuperación selectiva**. El modelo no necesita leer todo el documento en cada consulta; necesita acceder primero a una vista resumida y después cargar solo los fragmentos relevantes. Esto reduce coste, latencia y riesgo de perder información importante dentro de una ventana demasiado grande.


## Actividad 3 — API multimodal (stub)

Si tu curso usa un proveedor con visión, deja un **stub** que construya `messages` con una imagen (URL o path) + pregunta. Si no, comenta el formato esperado (`image_url`, etc.).


In [2]:
# Actividad 3 — API multimodal (stub)
# Estructura típica de un mensaje multimodal con texto + imagen.
# El formato exacto puede variar según el proveedor, pero muchos usan
# una lista de contenidos con type="text" y type="image_url".

image_url = "https://example.com/image.png"

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "Describe la imagen y extrae la información relevante."
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": image_url
                }
            }
        ]
    }
]

messages

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Describe la imagen y extrae la información relevante.'},
   {'type': 'image_url',
    'image_url': {'url': 'https://example.com/image.png'}}]}]